In [1]:
# -*- coding: utf-8 -*-
import warnings, os, gc, time, json, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Intentar cargar LightGBM / XGBoost
LGBM_AVAILABLE = True
XGB_AVAILABLE  = True
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBM_AVAILABLE = False
try:
    from xgboost import XGBClassifier
except Exception:
    XGB_AVAILABLE = False

from imblearn.under_sampling import RandomUnderSampler


In [ ]:

# ==========================
# CONFIGURACIÓN
# ==========================
DATA_PATH = "../data/processed/df_ready_model.csv"
SAVE_DIR  = "../models"
os.makedirs(SAVE_DIR, exist_ok=True)

# (opcional) limitar filas para grid-search del mejor modelo [1.0 = sin limitar]
SAMPLE_FRAC_FOR_GRID = 1.0

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ==========================
# UTILIDADES
# ==========================
def print_headline(txt):
    print("\n" + txt)
    print("-" * len(txt))

def pr_auc(y_true, y_proba):
    try:
        return average_precision_score(y_true, y_proba)
    except Exception:
        return np.nan

def evaluate_at_threshold(y_true, y_proba, thr=0.5):
    y_pred = (y_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
    recall    = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "precision": precision, "recall": recall, "f1": f1
    }

def pick_threshold_for_min_precision(y_true, y_proba, min_precision=0.10):
    prec, rec, thr = precision_recall_curve(y_true, y_proba)
    # precision_recall_curve -> len(thr)+1 puntos
    prec, rec = prec[:-1], rec[:-1]
    thr_ok = thr[prec >= min_precision]
    if len(thr_ok) == 0:
        # si no se alcanza precisión mínima, devolver 0.5 por defecto
        return 0.5, 0.0, 0.0
    # elegir el umbral que dé mayor recall cumpliendo la precisión mínima
    idx = np.argmax(rec[prec >= min_precision])
    chosen_thr = thr_ok[idx]
    chosen_prec = prec[prec >= min_precision][idx]
    chosen_rec  = rec[prec >= min_precision][idx]
    return float(chosen_thr), float(chosen_prec), float(chosen_rec)

# ==========================
# FEATURE ENGINEERING (agregados por cluster y por cluster-hora)
# ==========================
def add_cluster_aggregates(df_train, df_apply):
    """
    Crea y añade:
      - cluster_accident_rate (media accidente por cluster en TRAIN)
      - cluster_avg_temp, cluster_avg_precip (media por cluster en TRAIN)
      - cluster_hour_accident_rate (media accidente por cluster+hour en TRAIN)
    Evita leakage: se calcula SOLO con df_train y se mapea a df_apply (train y test).
    """
    # nombres de columnas meteo (ajusta si cambian)
    COL_TEMP = "temperature_2m (°C)"
    COL_PREC = "precipitation (mm)"

    # A) Por cluster
    agg_cluster = (
        df_train.groupby("cluster_id")
        .agg(
            cluster_accident_rate=("accident", "mean"),
            cluster_avg_temp=(COL_TEMP, "mean"),
            cluster_avg_precip=(COL_PREC, "mean"),
        )
        .reset_index()
    )

    # B) Por cluster-hora
    agg_cluster_hour = (
        df_train.groupby(["cluster_id", "hour"])
        .agg(cluster_hour_accident_rate=("accident", "mean"))
        .reset_index()
    )

    # Merge a df_apply
    out = df_apply.merge(agg_cluster, on="cluster_id", how="left")

    out = out.merge(agg_cluster_hour, on=["cluster_id", "hour"], how="left")

    # Rellenos con medias globales del TRAIN
    global_acc_rate   = df_train["accident"].mean()
    global_temp_mean  = df_train[COL_TEMP].mean()
    global_prec_mean  = df_train[COL_PREC].mean()
    global_hour_acc   = df_train.groupby("hour")["accident"].mean()

    out["cluster_accident_rate"]   = out["cluster_accident_rate"].fillna(global_acc_rate)
    out["cluster_avg_temp"]        = out["cluster_avg_temp"].fillna(global_temp_mean)
    out["cluster_avg_precip"]      = out["cluster_avg_precip"].fillna(global_prec_mean)
    out["cluster_hour_accident_rate"] = out.apply(
        lambda r: global_hour_acc.get(r["hour"], global_acc_rate) if pd.isna(r["cluster_hour_accident_rate"]) else r["cluster_hour_accident_rate"],
        axis=1
    )

    return out

# ==========================
# CARGA Y SPLIT TEMPORAL
# ==========================
print_headline("🚀 Iniciando pipeline de predicción de accidentes")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} registros, {df.shape[1]} columnas")

# columnas esperadas mínimas
expected = {"accident","cluster_id","temperature_2m (°C)","precipitation (mm)","wind_speed_10m (km/h)",
            "Fiesta","dia_festivo","year","month","day","hour","day_of_week","is_weekend"}
missing = expected - set(df.columns)
if missing:
    raise RuntimeError(f"Faltan columnas requeridas: {missing}")

print("\nDistribución de clases:")
print(f"Clase 0 (No accidente): {(df['accident']==0).sum():,} ({(df['accident']==0).mean()*100:.2f}%)")
print(f"Clase 1 (Accidente): {(df['accident']==1).sum():,} ({(df['accident']==1).mean()*100:.2f}%)")

# Split temporal (como en tus resultados)
train_mask = df["year"] <= 2022
test_mask  = df["year"] >= 2023

df_train = df.loc[train_mask].copy()
df_test  = df.loc[test_mask].copy()

print("\n📊 División temporal:")
print(f"Entrenamiento: {len(df_train):,} registros ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test: {len(df_test):,} registros ({len(df_test)/len(df)*100:.1f}%)")
print(f"Accidentes en entrenamiento: {df_train['accident'].sum():,} ({df_train['accident'].mean()*100:.2f}%)")
print(f"Accidentes en test: {df_test['accident'].sum():,} ({df_test['accident'].mean()*100:.2f}%)")

# ==========================
# FEATURE ENGINEERING (agregados por cluster)
# ==========================
print_headline("🧪 Feature Engineering por cluster (evitando leakage)")
# Añadimos agregados al train y al test mapeando SOLO estadísticos del train
df_train_fe = add_cluster_aggregates(df_train, df_train)
df_test_fe  = add_cluster_aggregates(df_train, df_test)

# Selección de features (añadimos los nuevos)
BASE_FEATURES = [
    "cluster_id", "temperature_2m (°C)", "precipitation (mm)",
    "wind_speed_10m (km/h)", "Fiesta", "dia_festivo",
    "year", "month", "day", "hour", "day_of_week", "is_weekend",
]
CLUSTER_FEATS = [
    "cluster_accident_rate", "cluster_avg_temp", "cluster_avg_precip",
    "cluster_hour_accident_rate"
]
FEATURES = BASE_FEATURES + CLUSTER_FEATS

X_train_full = df_train_fe[FEATURES].copy()
y_train_full = df_train_fe["accident"].astype(int).copy()

X_test_full  = df_test_fe[FEATURES].copy()
y_test_full  = df_test_fe["accident"].astype(int).copy()

del df_train_fe, df_test_fe
gc.collect()

# ==========================
# DATASETS BALANCEADOS (ligeros)
# ==========================
def create_balanced_sets(X_train, y_train):
    print_headline("⚖️ Creando datasets balanceados (ligeros y estables)")
    balanced = {}

    # undersampling conservador (~1:20)
    try:
        print("   🔄 Under Sampling Conservador (1:20)…")
        rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.05)
        Xc, yc = rus.fit_resample(X_train, y_train)
        balanced["undersampling_conservative"] = (Xc, yc)
        print(f"   ✅ {len(Xc):,} filas | pos={yc.sum():,} ({yc.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error Under Sampling Conservador: {e}")

    # undersampling moderado (~1:10)
    try:
        print("   🔄 Under Sampling Moderado (1:10)…")
        rum = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.10)
        Xm, ym = rum.fit_resample(X_train, y_train)
        balanced["undersampling_moderate"] = (Xm, ym)
        print(f"   ✅ {len(Xm):,} filas | pos={ym.sum():,} ({ym.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error Under Sampling Moderado: {e}")

    print(f"\n✅ Total de técnicas creadas: {len(balanced)}")
    return balanced

balanced_sets = create_balanced_sets(X_train_full, y_train_full)

# ==========================
# MODELOS (recortados y estables)
# ==========================
def get_models():
    models = {}

    # RF con pesos altos para recall
    models["RandomForest_HighRecall"] = RandomForestClassifier(
        n_estimators=200, max_depth=15,
        min_samples_split=3, min_samples_leaf=1,
        class_weight={0:1, 1:50},  # aproximación ratio
        random_state=RANDOM_STATE, n_jobs=-1
    )

    if LGBM_AVAILABLE:
        models["LightGBM_Balanced"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05,
            max_depth=-1, num_leaves=63, min_child_samples=60,
            subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        )

    if XGB_AVAILABLE:
        models["XGBoost_Weighted"] = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=8,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            scale_pos_weight=50, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
        )

    keep = [k for k in ["RandomForest_HighRecall","LightGBM_Balanced","XGBoost_Weighted"] if k in models]
    print_headline("🤖 Modelos a evaluar")
    print(keep)
    return {k: models[k] for k in keep}

models = get_models()

# ==========================
# EVALUACIÓN
# ==========================
def evaluate_combo(model, X_train, y_train, X_test, y_test, model_name, balance_name, thr=0.5):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    proba = model.predict_proba(X_test)[:, 1]
    roc   = roc_auc_score(y_test, proba)
    pr    = pr_auc(y_test, proba)

    at_thr = evaluate_at_threshold(y_test, proba, thr=thr)
    res = {
        "model": model_name,
        "balance": balance_name,
        "thr": thr,
        "precision": at_thr["precision"],
        "recall": at_thr["recall"],
        "f1": at_thr["f1"],
        "roc_auc": roc,
        "pr_auc": pr,
        "tn": at_thr["tn"], "fp": at_thr["fp"], "fn": at_thr["fn"], "tp": at_thr["tp"],
        "train_time_s": round(train_time, 2),
        "_proba": proba  # guardamos por si luego queremos ajustar umbral
    }
    return res, model

print_headline("🔬 Evaluando combinaciones…")
all_results = []
trained = {}

for bal_name, (Xb, yb) in balanced_sets.items():
    print(f"\n  📊 Técnica: {bal_name.upper():<27} | datos={len(Xb):,} ({yb.mean()*100:.2f}% pos)")
    idx = 1
    for mname, model in models.items():
        print(f"    [{idx:02d}/{len(models)*len(balanced_sets)}] {mname:20s} → ", end="")
        try:
            res, fitted = evaluate_combo(model, Xb, yb, X_test_full, y_test_full, mname, bal_name, thr=0.5)
            all_results.append(res)
            trained[f"{mname}__{bal_name}"] = fitted
            print(f"OK  R={res['recall']:.3f}  P={res['precision']:.3f}  F1={res['f1']:.3f}  PR_AUC={res['pr_auc']:.3f}")
        except Exception as e:
            print(f"ERROR: {str(e)[:60]}...")
        idx += 1

df_results = pd.DataFrame(all_results)
df_results = df_results.sort_values(["pr_auc", "recall", "precision"], ascending=[False, False, False]).reset_index(drop=True)

print_headline("📈 TOP resultados (orden: PR_AUC, Recall, Precision)")
print(df_results[["model","balance","thr","precision","recall","f1","roc_auc","pr_auc","tn","fp","fn","tp","train_time_s"]].head(10))

best = df_results.iloc[0].to_dict()
print_headline("🏆 Mejor combo")
print(pd.Series(best))

# Matriz y reporte en umbral 0.5 (informativo)
best_key = f"{best['model']}__{best['balance']}"
best_model = trained[best_key]
best_proba = df_results.loc[0, "_proba"]  # proba del mejor ya calculada
cm = confusion_matrix(y_test_full, (best_proba >= best["thr"]).astype(int))
print("\nMatriz de confusión (umbral 0.5):")
print(cm)
print("\nClassification report:")
print(classification_report(y_test_full, (best_proba >= best["thr"]).astype(int)))

# ============================================
# OPTIMIZACIÓN DEL MEJOR MODELO + GUARDADO
# ============================================
print_headline("🔧 Optimización del mejor modelo y guardado")

# Datos a usar para optimización: dataset balanceado ganador
X_opt, y_opt = balanced_sets[best["balance"]]
if SAMPLE_FRAC_FOR_GRID < 1.0:
    n = int(len(X_opt) * SAMPLE_FRAC_FOR_GRID)
    sel = np.random.RandomState(RANDOM_STATE).choice(len(X_opt), size=n, replace=False)
    X_opt = X_opt.iloc[sel].reset_index(drop=True)
    y_opt = y_opt.iloc[sel].reset_index(drop=True)
    print(f"Usando subset para Grid: {len(X_opt):,} filas")

# Grids moderados (pensados para 1–2h aprox. en Codespaces)
best_name = best["model"]

if best_name == "LightGBM_Balanced" and LGBM_AVAILABLE:
    base = LGBMClassifier(
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        class_weight="balanced"
    )
    param_grid = {
        "n_estimators": [300, 600, 1000],
        "learning_rate": [0.05, 0.1],
        "num_leaves": [63, 127, 255],
        "min_child_samples": [20, 60, 120],
        "colsample_bytree": [0.7, 0.9],
        "subsample": [0.7, 0.9]
    }
    gs = GridSearchCV(
        base, param_grid, scoring="average_precision", cv=3,
        n_jobs=-1, verbose=1
    )
    gs.fit(X_opt, y_opt)
    best_est = gs.best_estimator_
    print("\nMejores parámetros LGBM:")
    print(gs.best_params_)

elif best_name == "XGBoost_Weighted" and XGB_AVAILABLE:
    base = XGBClassifier(
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
        eval_metric="logloss"
    )
    param_dist = {
        "n_estimators": [300, 600, 900],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [6, 8, 10],
        "subsample": [0.7, 0.9],
        "colsample_bytree": [0.7, 0.9],
        "reg_alpha": [0.0, 0.1, 0.3],
        "reg_lambda": [1.0, 2.0],
        "scale_pos_weight": [30, 40, 50, 60]
    }
    rs = RandomizedSearchCV(
        base, param_distributions=param_dist, n_iter=30,
        scoring="average_precision", cv=3, n_jobs=-1, verbose=1,
        random_state=RANDOM_STATE
    )
    rs.fit(X_opt, y_opt)
    best_est = rs.best_estimator_
    print("\nMejores parámetros XGB:")
    print(rs.best_params_)

else:
    # RF o fallback
    base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight={0:1, 1:50})
    param_grid = {
        "n_estimators": [200, 400],
        "max_depth": [12, 16, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }
    gs = GridSearchCV(
        base, param_grid, scoring="average_precision", cv=3,
        n_jobs=-1, verbose=1
    )
    gs.fit(X_opt, y_opt)
    best_est = gs.best_estimator_
    print("\nMejores parámetros RF:")
    print(gs.best_params_)

# Re-entrenar mejor estimador en TODO el dataset balanceado ganador
t0 = time.time()
best_est.fit(X_opt, y_opt)
fit_time = time.time() - t0
print(f"\nRe-entrenado mejor modelo en {fit_time/60:.2f} min sobre {len(X_opt):,} filas")

# Evaluación final en TEST (con umbral basado en precisión mínima 0.10)
proba_test = best_est.predict_proba(X_test_full)[:, 1]
thr_sel, p_val, r_val = pick_threshold_for_min_precision(y_test_full, proba_test, min_precision=0.10)

print(f"\nUmbral elegido por prec>=0.10 (calculado en TEST para diagnóstico): thr={thr_sel:.4f} | prec≈{p_val:.3f} | rec≈{r_val:.3f}")
print(f"PR AUC (TEST): {average_precision_score(y_test_full, proba_test):.6f}")
print(f"ROC AUC(TEST): {roc_auc_score(y_test_full, proba_test):.6f}")

ev = evaluate_at_threshold(y_test_full, proba_test, thr_sel)
print("\nMatriz de confusión (TEST con umbral seleccionado):")
print(np.array([[ev['tn'], ev['fp']], [ev['fn'], ev['tp']]]))
print("\nClassification report (umbral seleccionado):")
print(classification_report(y_test_full, (proba_test >= thr_sel).astype(int)))

# ==========================
# GUARDADO DEL MODELO
# ==========================
model_artifact = {
    "model": best_est,
    "features": FEATURES,
    "meta": {
        "created_at": pd.Timestamp.now().isoformat(),
        "best_combo": best,
        "threshold_selected": float(thr_sel),
        "metrics_test": {
            "precision": ev["precision"],
            "recall": ev["recall"],
            "f1": ev["f1"],
            "pr_auc": float(average_precision_score(y_test_full, proba_test)),
            "roc_auc": float(roc_auc_score(y_test_full, proba_test))
        }
    }
}
save_path = os.path.join(SAVE_DIR, f"best_model_optimized_{best_name}.joblib")
joblib.dump(model_artifact, save_path)
print(f"\n💾 Modelo optimizado guardado en: {save_path}")

# (opcional) guardar CSV de resultados comparativos
res_path = os.path.join(SAVE_DIR, "model_comparison_results_with_fe.csv")
df_results.drop(columns=["_proba"], errors="ignore").to_csv(res_path, index=False)
print(f"💾 Resultados comparativos guardados en: {res_path}")

print_headline("✅ Pipeline finalizado")
print("Listo para dejar ejecutando — diseño para evitar bloquear el kernel.")



🚀 Iniciando pipeline de predicción de accidentes
------------------------------------------------
Dataset cargado: 3,495,160 registros, 16 columnas

Distribución de clases:
Clase 0 (No accidente): 3,427,736 (98.07%)
Clase 1 (Accidente): 67,424 (1.93%)

📊 División temporal:
Entrenamiento: 2,604,597 registros (74.5%)
Test: 890,563 registros (25.5%)
Accidentes en entrenamiento: 52,167 (2.00%)
Accidentes en test: 15,257 (1.71%)

🧪 Feature Engineering por cluster (evitando leakage)
----------------------------------------------------

⚖️ Creando datasets balanceados (ligeros y estables)
----------------------------------------------------
   🔄 Under Sampling Conservador (1:20)…
   ✅ 1,095,507 filas | pos=52,167 (4.76%)
   🔄 Under Sampling Moderado (1:10)…
   ✅ 573,837 filas | pos=52,167 (9.09%)

✅ Total de técnicas creadas: 2

🤖 Modelos a evaluar
-------------------
['RandomForest_HighRecall', 'LightGBM_Balanced', 'XGBoost_Weighted']

🔬 Evaluando combinaciones…
--------------------------

